In [ ]:

# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

#import numpy as np # linear algebra
#import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

#import os
#for dirname, _, filenames in os.walk('/kaggle/input'):
 #   for filename in filenames:
  #      print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session


In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import pandas as pd
import os
from torchvision import transforms
from sklearn.model_selection import train_test_split
import torch.nn as nn
import numpy as np

In [ ]:
class XrayDataset(Dataset):
    def __init__(self, df, img_dir, transform=None):
        self.data_frame = df
        self.img_dir = img_dir
        self.transform = transform
        self.label_columns = [col for col in self.data_frame.columns if col != 'id']

    def __len__(self):
        return len(self.data_frame)

    def __getitem__(self,idx):
        img_id = self.data_frame.iloc[idx]['id']
        img_name = os.path.join(self.img_dir,img_id)

        image = Image.open(img_name).convert('RGB')
        
        labels_vector = self.data_frame.iloc[idx][self.label_columns].values.astype('float32')
        label_idx = np.argmax(labels_vector)
        labels = torch.tensor(label_idx, dtype=torch.long)
        if self.transform:
            image = self.transform(image)
        return image, labels
        

In [ ]:
train_df = pd.read_csv("/kaggle/input/competitions/26-t-1-dl-gen-ainppe-1/train.csv")
DATA_DIR = "/kaggle/input/competitions/26-t-1-dl-gen-ainppe-1"
IMG_DIR = os.path.join(DATA_DIR, "images")

In [ ]:
train_data, val_data = train_test_split(train_df, test_size=0.20, random_state=2021)
print(train_data.size)
print(val_data.size)

In [ ]:
train_transforms = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
])
val_transforms = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
])

## Train Loader

In [ ]:
train_dataset = XrayDataset(
    df=train_data,
    img_dir = IMG_DIR,
    transform = train_transforms
)

train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True,
    num_workers=4
)

## Validation Loader

In [ ]:
val_dataset = XrayDataset(
    df=val_data,
    img_dir = IMG_DIR,
    transform = val_transforms
)

val_loader = DataLoader(
    val_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=4
)

In [ ]:
label_col = [col for col in train_df.columns if col != 'id']
print(label_col)

## Set Device

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

## Initializing Loss Function

In [ ]:
criterion = torch.nn.CrossEntropyLoss()

## Model Template Code

In [ ]:
import torchvision.models as models
import torch.nn as nn

model = models.densenet121(weights='DEFAULT')

num_ftrs = model.classifier.in_features
model.classifier = nn.Sequential(
    nn.Linear(num_ftrs, 512),
    nn.ReLU(),
    nn.Dropout(0.5),
    nn.Linear(512, 20)
)

model = model.to(device)


## Adding Optimizer

In [ ]:
import torch.optim as optim

optimizer = optim.Adam(model.parameters(), lr=1e-4)

In [ ]:
best_val_loss = float('inf')
best_model_path = "best_model.pth"

num_epochs = 5 # Start with 5-10

for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    
    for images, labels in train_loader:
        # Move data to GPU
        images = images.to(device)
        labels = labels.to(device)
        
        # Zero the gradients
        optimizer.zero_grad()
        
        # Forward pass
        outputs = model(images)
        loss = criterion(outputs, labels)
        
        # Backward pass (Calculates gradients)
        loss.backward()
        
        # Update weights
        optimizer.step()
        running_loss += loss.item()
    
    # Calculate Average Loss for the epoch
    epoch_loss = running_loss / len(train_loader)
    
    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            val_loss += loss.item()
            
    avg_val_loss = val_loss / len(val_loader)
    
    print(f"Epoch [{epoch+1}/{num_epochs}] | Train Loss: {epoch_loss:.4f} | Val Loss: {avg_val_loss:.4f}")
    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        torch.save(model.state_dict(), best_model_path)
        print("Best model saved!")

In [ ]:
# Load best model
model.load_state_dict(torch.load("best_model.pth"))

In [ ]:
TEST_CSV = os.path.join(DATA_DIR, "test.csv")
SUBMISSION_PATH = "/kaggle/working/submission.csv"

# Load test data
test_df = pd.read_csv(TEST_CSV)

class TestDataset(Dataset):
    def __init__(self, df, img_dir, transform=None):
        self.df = df
        self.img_dir = img_dir
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_path = os.path.join(self.img_dir, row["id"])
        
        image = Image.open(img_path).convert("RGB")

        if self.transform:
            image = self.transform(image)

        return image, row["id"]

# Dataset & loader
test_dataset = TestDataset(test_df, IMG_DIR, transform = val_transforms)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False, num_workers=2)


disease_threshold = 0.2
no_finding_threshold = 0.5
thresh_array = np.array([disease_threshold] * 19 + [no_finding_threshold])

# Inference (multi-class -> one-hot submission)
model.eval()
records = []


with torch.no_grad():
    for images, ids in test_loader:
        images = images.to(device)
        outputs = model(images)
        probs = torch.softmax(outputs, dim=1).cpu().numpy()
        probs[:, :-1] *= 1.4   # boost diseases
        probs[:, -1] *= 0.3    # suppress "No Finding"
   
        pred_class = np.argmax(probs, axis=1)

        for i in range(len(ids)):
            row = {"id": ids[i]}
            for j, col in enumerate(label_col):
                row[col] = int(pred_class[i] == j)
            records.append(row)

# Submission
submission = pd.DataFrame(records)
submission = submission[["id"] + list(label_col)]

submission.to_csv(SUBMISSION_PATH, index=False)

print("Saved:", SUBMISSION_PATH)
submission.head()